## Setup
Thin `exp_015f` calibration-refresh submit path with timer-driven watchdogs and early-safe `submission.csv` snapshots.

Expected Kaggle inputs:
- competition data `BirdCLEF+ 2026`
- TensorFlow 2.20 wheels dataset
- `perch_v2_cpu` model dataset
- refresh artifact dataset for `exp_015f`


In [ ]:
# Cell 0b — Kaggle input hints
TF_WHEELS_HINT = None  # e.g. "bc26-tensorflow-2-20-0"
COMPETITION_HINT = None  # e.g. "birdclef-2026"
PERCH_MODEL_HINT = None  # e.g. "perch_v2_cpu"
PERCH_CACHE_HINT = None  # not required for thin-submit; can stay None
ARTIFACTS_HINT = None  # e.g. "birdclef-exp015f-v18-calibration-refresh-artifacts"

SUBMIT_TIMER_ENABLED = True
SUBMIT_MAX_WALL_SECONDS = 4800
SUBMIT_RESERVE_SECONDS = 180
SUBMIT_SAVE_EARLY_SNAPSHOTS = False
SUBMIT_STAGE_REQUIREMENTS = {
    "isotonic_calibration": 300,
    "file_level_scaling": 120,
    "rank_aware_scaling": 90,
    "adaptive_delta_smooth": 90,
    "threshold_sharpening": 60,
}

print({
    "TF_WHEELS_HINT": TF_WHEELS_HINT,
    "COMPETITION_HINT": COMPETITION_HINT,
    "PERCH_MODEL_HINT": PERCH_MODEL_HINT,
    "PERCH_CACHE_HINT": PERCH_CACHE_HINT,
    "ARTIFACTS_HINT": ARTIFACTS_HINT,
    "SUBMIT_TIMER_ENABLED": SUBMIT_TIMER_ENABLED,
    "SUBMIT_MAX_WALL_SECONDS": SUBMIT_MAX_WALL_SECONDS,
    "SUBMIT_RESERVE_SECONDS": SUBMIT_RESERVE_SECONDS,
})


In [ ]:
# Cell 0 — Install TF 2.20 from attached Kaggle wheels
from pathlib import Path
import subprocess, sys

INPUT_ROOT = Path("/kaggle/input")

def find_single_file(pattern, hint=None):
    candidates = []
    for p in INPUT_ROOT.rglob(pattern):
        sp = str(p).lower()
        if hint is None or hint.lower() in sp:
            candidates.append(p)
    if not candidates and hint is not None:
        for p in INPUT_ROOT.rglob(pattern):
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError(f"Could not find {pattern!r} under /kaggle/input")
    candidates = sorted(set(candidates))
    return candidates[0]

tb_whl = find_single_file("tensorboard-2.20.0-*.whl", TF_WHEELS_HINT)
tf_whl = find_single_file("tensorflow-2.20.0-*.whl", TF_WHEELS_HINT)
print("TensorBoard wheel:", tb_whl)
print("TensorFlow wheel:", tf_whl)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(tb_whl)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(tf_whl)], check=True)


# BirdCLEF+ 2026 -- V18 Calibration Refresh Timed Submit

Operationalized `exp_015f` thin submit path with timer-aware watchdogs, stage logging, and early-safe fallback snapshots.


## Configuration
Mode switch and hyperparameters.


In [ ]:
# Cell 1 — Mode switch
MODE = "submit" 

assert MODE in {"train", "submit"}

print("MODE =", MODE)


In [ ]:
# Cell 2 — Imports and run config
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import gc
import json
import pickle
import re
import time
import warnings
from contextlib import nullcontext
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
try:
    from lightgbm import LGBMClassifier
    _LGBM_AVAILABLE = True
except ImportError:
    _LGBM_AVAILABLE = False
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
tf.experimental.numpy.experimental_enable_numpy_behavior()

_WALL_START = time.time()
INPUT_ROOT = Path("/kaggle/input")

def resolve_competition_dir():
    candidates = [
        Path("/kaggle/input/competitions/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026"),
    ]
    for p in candidates:
        if (p / "sample_submission.csv").exists() and (p / "taxonomy.csv").exists():
            return p
    found = []
    for p in INPUT_ROOT.iterdir():
        if not p.is_dir():
            continue
        if COMPETITION_HINT and COMPETITION_HINT.lower() not in str(p).lower():
            continue
        if (p / "sample_submission.csv").exists() and (p / "taxonomy.csv").exists():
            found.append(p)
    if not found:
        for p in INPUT_ROOT.rglob("sample_submission.csv"):
            parent = p.parent
            if (parent / "taxonomy.csv").exists():
                found.append(parent)
    if not found:
        raise FileNotFoundError("Could not resolve BirdCLEF competition directory under /kaggle/input")
    return sorted(found)[0]

def resolve_perch_model_dir():
    found = []
    for p in INPUT_ROOT.rglob("saved_model.pb"):
        parent = p.parent
        if (parent / "assets" / "labels.csv").exists():
            if PERCH_MODEL_HINT is None or PERCH_MODEL_HINT.lower() in str(parent).lower():
                found.append(parent)
    if not found:
        raise FileNotFoundError("Could not find perch_v2_cpu SavedModel under /kaggle/input")
    return sorted(found)[0]

def resolve_perch_cache_dir():
    found = []
    for p in INPUT_ROOT.rglob("full_perch_meta.parquet"):
        parent = p.parent
        if (parent / "full_perch_arrays.npz").exists():
            if PERCH_CACHE_HINT is None or PERCH_CACHE_HINT.lower() in str(parent).lower():
                found.append(parent)
    return sorted(found)[0] if found else None

BASE = resolve_competition_dir()
MODEL_DIR = resolve_perch_model_dir()
PERCH_CACHE_DIR = resolve_perch_cache_dir()
print("Competition dir:", BASE)
print("Perch model dir:", MODEL_DIR)
print("Perch cache dir:", PERCH_CACHE_DIR)


def resolve_artifacts_dir():
    found = []
    for p in INPUT_ROOT.rglob("artifacts_manifest.json"):
        parent = p.parent
        if ARTIFACTS_HINT is None or ARTIFACTS_HINT.lower() in str(parent).lower():
            found.append(parent)
    if not found:
        raise FileNotFoundError("Could not resolve V18 artifact dataset under /kaggle/input")
    return sorted(found)[0]

ARTIFACTS_DIR = resolve_artifacts_dir()
print("Artifacts dir:", ARTIFACTS_DIR)


SR = 32000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES = 60 * SR
N_WINDOWS = 12

# Kaggle code competitions effectively disable GPU submissions.
# Keep this thin submit notebook CPU-only for reproducible scoring.
DEVICE = torch.device("cpu")
AMP_ENABLED = False

if torch.cuda.is_available():
    raise RuntimeError(
        "Disable the accelerator in Kaggle notebook settings. "
        "For this competition, GPU-attached submissions are effectively limited to ~1 minute runtime."
    )


def autocast_context():
    return nullcontext()

def to_device_tensor(x, dtype):
    return torch.tensor(x, dtype=dtype, device=DEVICE)

def tensor_to_numpy(x):
    return x.detach().cpu().numpy()


LOGS = {}

TIMER_CFG = {
    "enabled": bool(SUBMIT_TIMER_ENABLED),
    "max_wall_seconds": float(SUBMIT_MAX_WALL_SECONDS),
    "reserve_seconds": float(SUBMIT_RESERVE_SECONDS),
    "save_early_snapshots": bool(SUBMIT_SAVE_EARLY_SNAPSHOTS),
    "stage_requirements": {str(k): float(v) for k, v in SUBMIT_STAGE_REQUIREMENTS.items()},
}

def wall_elapsed_seconds() -> float:
    return float(time.time() - _WALL_START)

def wall_remaining_seconds() -> float:
    return float(TIMER_CFG["max_wall_seconds"] - wall_elapsed_seconds())

def record_stage_event(stage_name: str, **kwargs):
    event = {"stage": stage_name, "elapsed_seconds": wall_elapsed_seconds()}
    event.update(kwargs)
    LOGS.setdefault("stage_events", []).append(event)
    extras = ", ".join(f"{k}={v}" for k, v in kwargs.items())
    suffix = f" | {extras}" if extras else ""
    print(f"[stage] {stage_name} | elapsed={event['elapsed_seconds']:.1f}s{suffix}")
    return event

def timer_allows(stage_name: str, required_seconds: float = 0.0) -> bool:
    elapsed = wall_elapsed_seconds()
    remaining = wall_remaining_seconds()
    reserve = float(TIMER_CFG["reserve_seconds"])
    allowed = (not TIMER_CFG["enabled"]) or (remaining > reserve + float(required_seconds))
    check = {
        "stage": stage_name,
        "elapsed_seconds": float(elapsed),
        "remaining_seconds": float(remaining),
        "required_seconds": float(required_seconds),
        "reserve_seconds": float(reserve),
        "allowed": bool(allowed),
    }
    LOGS.setdefault("timer_checks", []).append(check)
    print(
        f"[timer] {stage_name}: elapsed={elapsed:.1f}s remaining={remaining:.1f}s "
        f"required={required_seconds:.1f}s reserve={reserve:.1f}s allowed={allowed}"
    )
    return allowed

def build_submission_df(row_ids, probs):
    submission = pd.DataFrame(probs, columns=PRIMARY_LABELS)
    submission.insert(0, "row_id", np.asarray(row_ids))
    submission[PRIMARY_LABELS] = submission[PRIMARY_LABELS].astype(np.float32)
    return submission

def save_submission_snapshot(row_ids, probs, tag: str, make_primary: bool = False):
    submission = build_submission_df(row_ids, probs)
    out_path = Path('/kaggle/working') / f'submission_{tag}.csv'
    submission.to_csv(out_path, index=False)
    if make_primary:
        submission.to_csv('submission.csv', index=False)
        print(f"Saved primary submission.csv from snapshot: {tag}")
    snap = {
        "tag": str(tag),
        "elapsed_seconds": wall_elapsed_seconds(),
        "rows": int(len(submission)),
        "mean": float(np.asarray(probs).mean()),
        "min": float(np.asarray(probs).min()),
        "max": float(np.asarray(probs).max()),
        "primary": bool(make_primary),
        "path": str(out_path),
    }
    LOGS.setdefault("submission_snapshots", []).append(snap)
    return submission

CFG = {
    "mode": MODE,
    "verbose": MODE == "train",
    "run_oof_baseline": MODE == "train",
    "run_probe_check": False,
    "run_probe_grid": False,
    "batch_files": 16,
    "proxy_reduce_grid": ["max", "mean"],
    "proxy_reduce": "max",
    "run_proxy_reduce_grid": False,
    "dryrun_n_files": 50 if MODE == "train" else 20,
    "require_full_cache_in_submit": True,
    "full_cache_input_dir": PERCH_CACHE_DIR if PERCH_CACHE_DIR is not None else Path("/kaggle/input/perch-meta"),
    "full_cache_work_dir": Path("/kaggle/working/perch_cache"),
    "best_fusion": {"lambda_event": 0.45, "lambda_texture": 1.1, "lambda_proxy_texture": 0.9, "smooth_texture": 0.35, "smooth_event": 0.15},
    "proto_ssm": {"d_model": 320, "d_state": 32, "n_ssm_layers": 4, "dropout": 0.12, "n_prototypes": 2, "n_sites": 20, "meta_dim": 24, "use_cross_attn": True, "cross_attn_heads": 8},
    "proto_ssm_train": {"n_epochs": 80, "lr": 8e-4, "weight_decay": 1e-3, "val_ratio": 0.15, "patience": 20, "pos_weight_cap": 25.0, "distill_weight": 0.15, "proto_margin": 0.15, "label_smoothing": 0.03, "oof_n_splits": 5, "mixup_alpha": 0.4, "focal_gamma": 2.5, "swa_start_frac": 0.65, "swa_lr": 4e-4, "use_cosine_restart": True, "restart_period": 20},
    "frozen_best_probe": {"pca_dim": 128, "min_pos": 5, "C": 0.75, "alpha": 0.45},
    "residual_ssm": {"d_model": 128, "d_state": 16, "n_ssm_layers": 2, "dropout": 0.1, "correction_weight": 0.35, "n_epochs": 15, "lr": 8e-4, "patience": 12},
    "temperature": {"aves": 1.10, "texture": 0.95},
    "file_level_top_k": 2,
    "tta_shifts": [0],
    "rank_aware_scale": True,
    "rank_aware_power": 0.4,
    "delta_shift_alpha": 0.20,
    "threshold_grid": np.array([0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70], dtype=np.float32),
    "probe_backend": "mlp",
    "mlp_params": {
        "hidden_layer_sizes": (256, 128),
        "activation": "relu",
        "max_iter": 500,
        "early_stopping": True,
        "validation_fraction": 0.15,
        "n_iter_no_change": 20,
        "random_state": 42,
        "learning_rate_init": 5e-4,
        "alpha": 0.005,
    },
    "lgbm_params": {
        "n_estimators": 200,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_samples": 20,
        "objective": "binary",
        "verbosity": -1,
        "random_state": 42,
    },
}
CFG["full_cache_work_dir"].mkdir(parents=True, exist_ok=True)
print("Perch work cache dir:", CFG["full_cache_work_dir"])
print("Torch device:", DEVICE, "(forced CPU for code-competition safety)")

try:
    tf.config.set_visible_devices([], "GPU")
except Exception:
    pass

print("Loaded V18 timed calibration-refresh submit config for exp_015h")
print("Timer config:", TIMER_CFG)


## Data Loading & Preprocessing
Load competition data, parse labels, build truth matrices.


In [ ]:
taxonomy = pd.read_csv(BASE / "taxonomy.csv")
sample_sub = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)

taxonomy["primary_label"] = taxonomy["primary_label"].astype(str)
soundscape_labels["primary_label"] = soundscape_labels["primary_label"].astype(str)

def parse_soundscape_labels(x):
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(";") if t.strip()]

FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return {
            "file_id": None,
            "site": None,
            "date": pd.NaT,
            "time_utc": None,
            "hour_utc": -1,
            "month": -1,
        }
    file_id, site, ymd, hms = m.groups()
    dt = pd.to_datetime(ymd, format="%Y%m%d", errors="coerce")
    return {
        "file_id": file_id,
        "site": site,
        "date": dt,
        "time_utc": hms,
        "hour_utc": int(hms[:2]),
        "month": int(dt.month) if pd.notna(dt) else -1,
    }

def union_labels(series):
    return sorted(set(lbl for x in series for lbl in parse_soundscape_labels(x)))

# Deduplicate duplicated rows and aggregate labels per 5s window
sc_clean = (
    soundscape_labels
    .groupby(["filename", "start", "end"])["primary_label"]
    .apply(union_labels)
    .reset_index(name="label_list")
)

sc_clean["start_sec"] = pd.to_timedelta(sc_clean["start"]).dt.total_seconds().astype(int)
sc_clean["end_sec"] = pd.to_timedelta(sc_clean["end"]).dt.total_seconds().astype(int)
sc_clean["row_id"] = sc_clean["filename"].str.replace(".ogg", "", regex=False) + "_" + sc_clean["end_sec"].astype(str)

meta = sc_clean["filename"].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, meta], axis=1)

# Fully-labeled files
windows_per_file = sc_clean.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc_clean["file_fully_labeled"] = sc_clean["filename"].isin(full_files)

# Multi-hot label matrix aligned with sc_clean row order
label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}
Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)

for i, labels in enumerate(sc_clean["label_list"]):
    idxs = [label_to_idx[lbl] for lbl in labels if lbl in label_to_idx]
    if idxs:
        Y_SC[i, idxs] = 1

full_truth = (
    sc_clean[sc_clean["file_fully_labeled"]]
    .sort_values(["filename", "end_sec"])
    .reset_index(drop=False)
)

Y_FULL_TRUTH = Y_SC[full_truth["index"].to_numpy()]

print("sc_clean:", sc_clean.shape)
print("Y_SC:", Y_SC.shape, Y_SC.dtype)
print("Full files:", len(full_files))
print("Trusted full windows:", len(full_truth))
print("Active classes in full windows:", int((Y_FULL_TRUTH.sum(axis=0) > 0).sum()))


## Model & Mapping
Load Perch v2 model and build species-to-BirdClassifier mapping.


In [ ]:
# Cell 3 — Load Perch, mapping, and selective frog proxies
BEST = CFG["best_fusion"]
birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn = birdclassifier.signatures["serving_default"]

bc_labels = (
    pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
    .reset_index()
    .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
)

NO_LABEL_INDEX = len(bc_labels)

MANUAL_SCIENTIFIC_NAME_MAP = {
    # Optional future synonym fixes (add manual name corrections here)
}

taxonomy = taxonomy.copy()
taxonomy["scientific_name_lookup"] = taxonomy["scientific_name"].replace(MANUAL_SCIENTIFIC_NAME_MAP)

bc_lookup = bc_labels.rename(columns={"scientific_name": "scientific_name_lookup"})

mapping = taxonomy.merge(
    bc_lookup[["scientific_name_lookup", "bc_index"]],
    on="scientific_name_lookup",
    how="left"
)

mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL_INDEX).astype(int)

label_to_bc_index = mapping.set_index("primary_label")["bc_index"]
BC_INDICES = np.array([int(label_to_bc_index.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)

MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA = {"Amphibia", "Insecta"}

ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in np.where(Y_SC.sum(axis=0) > 0)[0]]

idx_active_texture = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) in TEXTURE_TAXA],
    dtype=np.int32
)
idx_active_event = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) not in TEXTURE_TAXA],
    dtype=np.int32
)

idx_mapped_active_texture = idx_active_texture[MAPPED_MASK[idx_active_texture]]
idx_mapped_active_event = idx_active_event[MAPPED_MASK[idx_active_event]]

idx_unmapped_active_texture = idx_active_texture[~MAPPED_MASK[idx_active_texture]]
idx_unmapped_active_event = idx_active_event[~MAPPED_MASK[idx_active_event]]

idx_unmapped_inactive = np.array(
    [i for i in UNMAPPED_POS if PRIMARY_LABELS[i] not in ACTIVE_CLASSES],
    dtype=np.int32
)

# Build automatic genus proxies for unmapped non-sonotypes
unmapped_df = mapping[mapping["bc_index"] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[
    ~unmapped_df["primary_label"].astype(str).str.contains("son", na=False)
].copy()

def get_genus_hits(scientific_name):
    genus = str(scientific_name).split()[0]
    hits = bc_labels[
        bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)
    ].copy()
    return genus, hits

proxy_map = {}
for _, row in unmapped_non_sonotype.iterrows():
    target = row["primary_label"]
    sci = row["scientific_name"]
    genus, hits = get_genus_hits(sci)
    if len(hits) > 0:
        proxy_map[target] = {
            "target_scientific_name": sci,
            "genus": genus,
            "bc_indices": hits["bc_index"].astype(int).tolist(),
            "proxy_scientific_names": hits["scientific_name"].tolist(),
        }

# Enable genus proxies for Amphibia, Insecta, and Aves (unmapped species)
PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
SELECTED_PROXY_TARGETS = sorted([
    t for t in proxy_map.keys()
    if CLASS_NAME_MAP.get(t) in PROXY_TAXA
])
print(f"Proxy targets by class: { {cls: sum(1 for t in SELECTED_PROXY_TARGETS if CLASS_NAME_MAP.get(t)==cls) for cls in PROXY_TAXA} }")

selected_proxy_pos = np.array([label_to_idx[c] for c in SELECTED_PROXY_TARGETS], dtype=np.int32)

selected_proxy_pos_to_bc = {
    label_to_idx[target]: np.array(proxy_map[target]["bc_indices"], dtype=np.int32)
    for target in SELECTED_PROXY_TARGETS
}

idx_selected_proxy_active_texture = np.intersect1d(selected_proxy_pos, idx_active_texture)
idx_selected_prioronly_active_texture = np.setdiff1d(idx_unmapped_active_texture, selected_proxy_pos)
idx_selected_prioronly_active_event = np.setdiff1d(idx_unmapped_active_event, selected_proxy_pos)

print(f"Mapped classes: {MAPPED_MASK.sum()} / {N_CLASSES}")
print(f"Unmapped classes: {(~MAPPED_MASK).sum()}")
print("Selected frog proxy targets:", SELECTED_PROXY_TARGETS)
print("Active texture classes:", len(idx_active_texture))
print("Selected proxy active texture:", len(idx_selected_proxy_active_texture))
print("Prior-only active texture:", len(idx_selected_prioronly_active_texture))
print("Prior-only active event:", len(idx_selected_prioronly_active_event))


## Utilities
Metrics, smoothing, and feature helpers.


In [ ]:
# Cell 4 — Metrics and helper utilities
def macro_auc_skip_empty(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")

def smooth_cols_fixed12(scores, cols, alpha=0.35):
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()

    s = scores.copy()
    assert len(s) % N_WINDOWS == 0, "Expected full-file blocks of 12 windows"
    view = s.reshape(-1, N_WINDOWS, s.shape[1])

    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)

    view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return s

def smooth_events_fixed12(scores, cols, alpha=0.15):
    """Soft max-pool context for event birds (Aves).
    Uses local_max instead of average neighbor, preserving transient call detection."""
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s = scores.copy()
    assert len(s) % N_WINDOWS == 0
    view = s.reshape(-1, N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    local_max = np.maximum(x, np.maximum(prev_x, next_x))
    view[:, :, cols] = (1.0 - alpha) * x + alpha * local_max
    return s

def seq_features_1d(v):
    """
    v: shape (n_rows,), ordered as full-file blocks of 12 windows
    Extended: tambah std_v untuk capture variance temporal dalam file
    """
    assert len(v) % N_WINDOWS == 0, "Expected full-file blocks of 12 windows"
    x = v.reshape(-1, N_WINDOWS)

    prev_v = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    next_v = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    mean_v = np.repeat(x.mean(axis=1), N_WINDOWS)
    max_v  = np.repeat(x.max(axis=1),  N_WINDOWS)
    std_v  = np.repeat(x.std(axis=1),  N_WINDOWS)

    return prev_v, next_v, mean_v, max_v, std_v


In [ ]:
# V16/V17 NEW: Focal loss, file-level scaling, TTA, rank-aware, delta shift, per-class thresholds

def focal_bce_with_logits(logits, targets, gamma=2.0, pos_weight=None, reduction="mean"):
    """Focal loss for multi-label classification.
    Reduces contribution of easy examples, focuses on hard ones."""
    if pos_weight is not None:
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=pos_weight, reduction="none"
        )
    else:
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    
    p = torch.sigmoid(logits)
    pt = targets * p + (1 - targets) * (1 - p)
    focal_weight = (1 - pt) ** gamma
    loss = focal_weight * bce
    
    if reduction == "mean":
        return loss.mean()
    return loss


def file_level_confidence_scale(preds, n_windows=12, top_k=2):
    """Rank 1/2 technique: scale each window's predictions by the file's top-K mean confidence."""
    N, C = preds.shape
    assert N % n_windows == 0
    view = preds.reshape(-1, n_windows, C)
    sorted_view = np.sort(view, axis=1)
    top_k_mean = sorted_view[:, -top_k:, :].mean(axis=1, keepdims=True)
    scaled = view * top_k_mean
    return scaled.reshape(N, C)


def temporal_shift_tta(emb_files, logits_files, model, site_ids, hours, shifts=[0, 1, -1]):
    """TTA by circular-shifting the 12-window embedding sequence."""
    all_preds = []
    model.eval()
    
    for shift in shifts:
        if shift == 0:
            e = emb_files
            l = logits_files
        else:
            e = np.roll(emb_files, shift, axis=1)
            l = np.roll(logits_files, shift, axis=1)
        
        with torch.no_grad():
            with autocast_context():
                out, _, _ = model(
                    to_device_tensor(e, torch.float32),
                    to_device_tensor(l, torch.float32),
                    site_ids=to_device_tensor(site_ids, torch.long),
                    hours=to_device_tensor(hours, torch.long),
                )
            pred = tensor_to_numpy(out)
        
        if shift != 0:
            pred = np.roll(pred, -shift, axis=1)
        
        all_preds.append(pred)
    
    return np.mean(all_preds, axis=0)


# V17: Post-processing utilities

def rank_aware_scaling(scores, n_windows=12, power=0.5):
    """V17: 2025 Rank 3 technique. Scale each window by (file_max)^power.
    Suppresses predictions in uncertain files, boosts confident files."""
    N, C = scores.shape
    assert N % n_windows == 0
    n_files = N // n_windows
    
    view = scores.reshape(n_files, n_windows, C)
    file_max = view.max(axis=1, keepdims=True)  # (F, 1, C)
    
    # Apply power transform to file max
    scale = np.power(file_max, power)
    
    # Scale each window
    scaled = view * scale
    return scaled.reshape(N, C)


def delta_shift_smooth(scores, n_windows=12, alpha=0.15):
    """V17: 2025 Rank 1 technique. Temporal smoothing across windows.
    new[t] = (1-alpha)*old[t] + 0.5*alpha*(old[t-1] + old[t+1])"""
    N, C = scores.shape
    assert N % n_windows == 0
    n_files = N // n_windows
    
    view = scores.reshape(n_files, n_windows, C)
    
    # Create shifted versions
    prev_view = np.concatenate([view[:, :1, :], view[:, :-1, :]], axis=1)
    next_view = np.concatenate([view[:, 1:, :], view[:, -1:, :]], axis=1)
    
    # Delta shift smoothing
    smoothed = (1 - alpha) * view + 0.5 * alpha * (prev_view + next_view)
    
    return smoothed.reshape(N, C)


def optimize_per_class_thresholds(oof_scores, y_true, n_windows=12, thresholds=[0.3, 0.4, 0.5, 0.6, 0.7]):
    """V17: Find optimal decision threshold per class from OOF predictions.
    Optimizes F1-like metric (precision-recall balance) for each species."""
    n_classes = oof_scores.shape[1]
    best_thresholds = np.zeros(n_classes)
    best_scores = np.zeros(n_classes)
    
    for c in range(n_classes):
        y_c = y_true[:, c]
        scores_c = oof_scores[:, c]
        
        # Skip classes with no positive samples
        if y_c.sum() == 0:
            best_thresholds[c] = 0.5
            continue
            
        # Find best threshold
        best_f1 = 0
        best_t = 0.5
        
        for t in thresholds:
            pred_c = (scores_c > t).astype(int)
            tp = ((pred_c == 1) & (y_c == 1)).sum()
            fp = ((pred_c == 1) & (y_c == 0)).sum()
            fn = ((pred_c == 0) & (y_c == 1)).sum()
            
            if tp + fp == 0 or tp + fn == 0:
                continue
                
            precision = tp / (tp + fp)
            recall = tp / (tp + fn)
            f1 = 2 * precision * recall / (precision + recall + 1e-8)
            
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
        
        best_thresholds[c] = best_t
        best_scores[c] = best_f1
    
    return best_thresholds, best_scores


def apply_per_class_thresholds(scores, thresholds, n_windows=12):
    """V17: Apply per-class thresholds to convert scores to binary predictions."""
    N, C = scores.shape
    assert C == len(thresholds)
    
    # For competition, we submit probabilities but threshold for metrics
    # Apply threshold as a scaling factor that sharpens confident predictions
    scaled = np.copy(scores)
    
    for c in range(C):
        t = thresholds[c]
        # Sharpen: push above-threshold scores higher, below-threshold lower
        mask_above = scores[:, c] > t
        scaled[mask_above, c] = 0.5 + 0.5 * (scores[mask_above, c] - t) / (1 - t + 1e-8)
        scaled[~mask_above, c] = 0.5 * scores[~mask_above, c] / (t + 1e-8)
    
    return np.clip(scaled, 0, 1)


print("V17 utilities defined: focal_bce_with_logits, file_level_confidence_scale, temporal_shift_tta,")
print("  rank_aware_scaling, delta_shift_smooth, optimize_per_class_thresholds, apply_per_class_thresholds")


## Perch Inference Engine
Core inference function with embedding extraction and genus proxies.


In [ ]:
# Cell 5 — Perch inference with embeddings + selective proxies
def read_soundscape_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr != SR:
        raise ValueError(f"Unexpected sample rate {sr} in {path}; expected {SR}")
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    elif len(y) > FILE_SAMPLES:
        y = y[:FILE_SAMPLES]
    return y

def infer_perch_with_embeddings(paths, batch_files=16, verbose=True, proxy_reduce="max"):
    paths = [Path(p) for p in paths]
    n_files = len(paths)
    n_rows = n_files * N_WINDOWS

    row_ids = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites = np.empty(n_rows, dtype=object)
    hours = np.empty(n_rows, dtype=np.int16)

    scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, 1536), dtype=np.float32)

    write_row = 0
    iterator = range(0, n_files, batch_files)
    if verbose:
        iterator = tqdm(iterator, total=(n_files + batch_files - 1) // batch_files, desc="Perch batches")

    for start in iterator:
        batch_paths = paths[start:start + batch_files]
        batch_n = len(batch_paths)

        x = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
        batch_row_start = write_row
        x_pos = 0

        for path in batch_paths:
            y = read_soundscape_60s(path)
            x[x_pos:x_pos + N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)

            meta = parse_soundscape_filename(path.name)
            stem = path.stem

            row_ids[write_row:write_row + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
            filenames[write_row:write_row + N_WINDOWS] = path.name
            sites[write_row:write_row + N_WINDOWS] = meta["site"]
            hours[write_row:write_row + N_WINDOWS] = int(meta["hour_utc"])

            x_pos += N_WINDOWS
            write_row += N_WINDOWS

        outputs = infer_fn(inputs=tf.convert_to_tensor(x))
        logits = outputs["label"].numpy().astype(np.float32, copy=False)
        emb = outputs["embedding"].numpy().astype(np.float32, copy=False)

        scores[batch_row_start:write_row, MAPPED_POS] = logits[:, MAPPED_BC_INDICES]
        embeddings[batch_row_start:write_row] = emb

        # Selected frog proxies
        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            sub = logits[:, bc_idx_arr]
            if proxy_reduce == "max":
                proxy_score = sub.max(axis=1)
            elif proxy_reduce == "mean":
                proxy_score = sub.mean(axis=1)
            else:
                raise ValueError("proxy_reduce must be 'max' or 'mean'")
            scores[batch_row_start:write_row, pos] = proxy_score.astype(np.float32)

        del x, outputs, logits, emb
        gc.collect()

    meta_df = pd.DataFrame({
        "row_id": row_ids,
        "filename": filenames,
        "site": sites,
        "hour_utc": hours,
    })

    return meta_df, scores, embeddings


## Cache Management
Load pre-computed Perch outputs or compute from scratch.


In [ ]:
# Cell 7 — Fold-safe metadata prior tables
def fit_prior_tables(prior_df, Y_prior):
    prior_df = prior_df.reset_index(drop=True)

    global_p = Y_prior.mean(axis=0).astype(np.float32)

    # Site
    site_keys = sorted(prior_df["site"].dropna().astype(str).unique().tolist())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    site_p = np.zeros((len(site_keys), Y_prior.shape[1]), dtype=np.float32)

    for s in site_keys:
        i = site_to_i[s]
        mask = prior_df["site"].astype(str).values == s
        site_n[i] = mask.sum()
        site_p[i] = Y_prior[mask].mean(axis=0)

    # Hour
    hour_keys = sorted(prior_df["hour_utc"].dropna().astype(int).unique().tolist())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    hour_p = np.zeros((len(hour_keys), Y_prior.shape[1]), dtype=np.float32)

    for h in hour_keys:
        i = hour_to_i[h]
        mask = prior_df["hour_utc"].astype(int).values == h
        hour_n[i] = mask.sum()
        hour_p[i] = Y_prior[mask].mean(axis=0)

    # Site-hour
    sh_to_i = {}
    sh_n_list = []
    sh_p_list = []

    for (s, h), idx in prior_df.groupby(["site", "hour_utc"]).groups.items():
        sh_to_i[(str(s), int(h))] = len(sh_n_list)
        idx = np.array(list(idx))
        sh_n_list.append(len(idx))
        sh_p_list.append(Y_prior[idx].mean(axis=0))

    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if len(sh_p_list) else np.zeros((0, Y_prior.shape[1]), dtype=np.float32)

    return {
        "global_p": global_p,
        "site_to_i": site_to_i,
        "site_n": site_n,
        "site_p": site_p,
        "hour_to_i": hour_to_i,
        "hour_n": hour_n,
        "hour_p": hour_p,
        "sh_to_i": sh_to_i,
        "sh_n": sh_n,
        "sh_p": sh_p,
    }

def prior_logits_from_tables(sites, hours, tables, eps=1e-4):
    n = len(sites)
    p = np.repeat(tables["global_p"][None, :], n, axis=0).astype(np.float32, copy=True)

    site_idx = np.fromiter(
        (tables["site_to_i"].get(str(s), -1) for s in sites),
        dtype=np.int32,
        count=n
    )
    hour_idx = np.fromiter(
        (tables["hour_to_i"].get(int(h), -1) if int(h) >= 0 else -1 for h in hours),
        dtype=np.int32,
        count=n
    )
    sh_idx = np.fromiter(
        (tables["sh_to_i"].get((str(s), int(h)), -1) if int(h) >= 0 else -1 for s, h in zip(sites, hours)),
        dtype=np.int32,
        count=n
    )

    valid = hour_idx >= 0
    if valid.any():
        nh = tables["hour_n"][hour_idx[valid]][:, None]
        wh = nh / (nh + 8.0)
        p[valid] = wh * tables["hour_p"][hour_idx[valid]] + (1.0 - wh) * p[valid]

    valid = site_idx >= 0
    if valid.any():
        ns = tables["site_n"][site_idx[valid]][:, None]
        ws = ns / (ns + 8.0)
        p[valid] = ws * tables["site_p"][site_idx[valid]] + (1.0 - ws) * p[valid]

    valid = sh_idx >= 0
    if valid.any():
        nsh = tables["sh_n"][sh_idx[valid]][:, None]
        wsh = nsh / (nsh + 4.0)
        p[valid] = wsh * tables["sh_p"][sh_idx[valid]] + (1.0 - wsh) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)

def fuse_scores_with_tables(base_scores, sites, hours, tables,
                            lambda_event=BEST["lambda_event"],
                            lambda_texture=BEST["lambda_texture"],
                            lambda_proxy_texture=BEST["lambda_proxy_texture"],
                            smooth_texture=BEST["smooth_texture"],
                            smooth_event=BEST["smooth_event"]):
    scores = base_scores.copy()
    prior = prior_logits_from_tables(sites, hours, tables)

    # mapped active
    if len(idx_mapped_active_event):
        scores[:, idx_mapped_active_event] += lambda_event * prior[:, idx_mapped_active_event]

    if len(idx_mapped_active_texture):
        scores[:, idx_mapped_active_texture] += lambda_texture * prior[:, idx_mapped_active_texture]

    # selected frog proxies
    if len(idx_selected_proxy_active_texture):
        scores[:, idx_selected_proxy_active_texture] += lambda_proxy_texture * prior[:, idx_selected_proxy_active_texture]

    # prior-only active unmapped
    if len(idx_selected_prioronly_active_event):
        scores[:, idx_selected_prioronly_active_event] = lambda_event * prior[:, idx_selected_prioronly_active_event]

    if len(idx_selected_prioronly_active_texture):
        scores[:, idx_selected_prioronly_active_texture] = lambda_texture * prior[:, idx_selected_prioronly_active_texture]

    # inactive unmapped
    if len(idx_unmapped_inactive):
        scores[:, idx_unmapped_inactive] = -8.0

    scores = smooth_cols_fixed12(scores, idx_active_texture, alpha=smooth_texture)
    scores = smooth_events_fixed12(scores, idx_active_event, alpha=smooth_event)
    return scores.astype(np.float32, copy=False), prior


In [ ]:
# Cell 9 — Classwise embedding-probe helpers
def build_class_features(emb_proj, raw_col, prior_col, base_col):
    """
    emb_proj: (n, d)
    raw_col, prior_col, base_col: (n,)
    returns: (n, d + 13)

    Fitur: embedding + 7 sequential + 3 interaction + std + 3 diff
    """
    prev_base, next_base, mean_base, max_base, std_base = seq_features_1d(base_col)

    # Diff features: posisi window relatif terhadap konteks file
    diff_mean = base_col - mean_base   # apakah window ini lebih tinggi dari rata2 file?
    diff_prev = base_col - prev_base   # onset: naik dari window sebelumnya?
    diff_next = base_col - next_base   # offset: turun ke window berikutnya?

    feats = np.concatenate([
        emb_proj,
        raw_col[:, None],
        prior_col[:, None],
        base_col[:, None],
        prev_base[:, None],
        next_base[:, None],
        mean_base[:, None],
        max_base[:, None],
        std_base[:, None],             # variance temporal dalam file
        diff_mean[:, None],            # deviasi dari mean file
        diff_prev[:, None],            # deteksi onset
        diff_next[:, None],            # deteksi offset
        # interaction terms
        (raw_col * prior_col)[:, None],
        (raw_col * base_col)[:, None],
        (prior_col * base_col)[:, None],
    ], axis=1)

    return feats.astype(np.float32, copy=False)

def run_oof_embedding_probe(
    scores_raw,
    emb,
    meta_df,
    y_true,
    pca_dim=64,
    min_pos=8,
    C=0.25,
    alpha=0.5,
):
    groups = meta_df["filename"].to_numpy()
    gkf = GroupKFold(n_splits=5)

    oof_base_local = np.zeros_like(scores_raw, dtype=np.float32)
    oof_final = np.zeros_like(scores_raw, dtype=np.float32)

    modeled_counts = np.zeros(scores_raw.shape[1], dtype=np.int32)

    split_list = list(gkf.split(scores_raw, groups=groups))

    for fold, (tr_idx, va_idx) in enumerate(tqdm(split_list, desc="Embedding-probe folds", disable=not CFG["verbose"]), 1):
    # for fold, (tr_idx, va_idx) in enumerate(tqdm(split_list, desc="Embedding-probe folds"), 1):
        tr_idx = np.sort(tr_idx)
        va_idx = np.sort(va_idx)

        val_files = set(meta_df.iloc[va_idx]["filename"].tolist())

        # Fold-safe priors
        prior_mask = ~sc_clean["filename"].isin(val_files).values
        prior_df_fold = sc_clean.loc[prior_mask].reset_index(drop=True)
        Y_prior_fold = Y_SC[prior_mask]
        tables = fit_prior_tables(prior_df_fold, Y_prior_fold)

        base_tr, prior_tr = fuse_scores_with_tables(
            scores_raw[tr_idx],
            sites=meta_df.iloc[tr_idx]["site"].to_numpy(),
            hours=meta_df.iloc[tr_idx]["hour_utc"].to_numpy(),
            tables=tables,
        )
        base_va, prior_va = fuse_scores_with_tables(
            scores_raw[va_idx],
            sites=meta_df.iloc[va_idx]["site"].to_numpy(),
            hours=meta_df.iloc[va_idx]["hour_utc"].to_numpy(),
            tables=tables,
        )

        oof_base_local[va_idx] = base_va
        oof_final[va_idx] = base_va

        # Embedding preprocessing on train fold only
        scaler = StandardScaler()
        emb_tr_s = scaler.fit_transform(emb[tr_idx])
        emb_va_s = scaler.transform(emb[va_idx])

        n_comp = min(pca_dim, emb_tr_s.shape[0] - 1, emb_tr_s.shape[1])
        pca = PCA(n_components=n_comp)
        Z_tr = pca.fit_transform(emb_tr_s).astype(np.float32)
        Z_va = pca.transform(emb_va_s).astype(np.float32)

        class_iterator = np.where(y_true[tr_idx].sum(axis=0) >= min_pos)[0].tolist()

        for cls_idx in tqdm(class_iterator, desc=f"Fold {fold} classes", leave=False, disable=not CFG["verbose"]):
        # for cls_idx in tqdm(class_iterator, desc=f"Fold {fold} classes", leave=False):
            y_tr = y_true[tr_idx, cls_idx]

            if y_tr.sum() == 0 or y_tr.sum() == len(y_tr):
                continue

            X_tr_cls = build_class_features(
                Z_tr,
                raw_col=scores_raw[tr_idx, cls_idx],
                prior_col=prior_tr[:, cls_idx],
                base_col=base_tr[:, cls_idx],
            )
            X_va_cls = build_class_features(
                Z_va,
                raw_col=scores_raw[va_idx, cls_idx],
                prior_col=prior_va[:, cls_idx],
                base_col=base_va[:, cls_idx],
            )

            # Pilih backend probe: mlp | lgbm | logreg
            backend = CFG.get("probe_backend", "mlp")
            n_pos = int(y_tr.sum())
            n_neg = len(y_tr) - n_pos

            if backend == "mlp":
                # MLPClassifier tidak support sample_weight
                # Gunakan oversampling: duplikasi positif agar balance
                if n_pos > 0 and n_neg > n_pos:
                    repeat = max(1, n_neg // n_pos)
                    pos_idx = np.where(y_tr == 1)[0]
                    X_bal = np.vstack([X_tr_cls, np.tile(X_tr_cls[pos_idx], (repeat, 1))])
                    y_bal = np.concatenate([y_tr, np.ones(len(pos_idx) * repeat, dtype=y_tr.dtype)])
                else:
                    X_bal, y_bal = X_tr_cls, y_tr
                clf = MLPClassifier(**CFG["mlp_params"])
                clf.fit(X_bal, y_bal)
                pred_va = clf.predict_proba(X_va_cls)[:, 1].astype(np.float32)
                pred_va = np.log(pred_va + 1e-7) - np.log(1 - pred_va + 1e-7)
            elif backend == "lgbm" and _LGBM_AVAILABLE:
                scale_pos = max(1.0, n_neg / max(n_pos, 1))
                clf = LGBMClassifier(
                    **CFG["lgbm_params"],
                    scale_pos_weight=scale_pos,
                )
                clf.fit(X_tr_cls, y_tr)
                pred_va = clf.predict_proba(X_va_cls)[:, 1].astype(np.float32)
                pred_va = np.log(pred_va + 1e-7) - np.log(1 - pred_va + 1e-7)
            else:
                clf = LogisticRegression(
                    C=C, max_iter=400, solver="liblinear",
                    class_weight="balanced",
                )
                clf.fit(X_tr_cls, y_tr)
                pred_va = clf.decision_function(X_va_cls).astype(np.float32)

            oof_final[va_idx, cls_idx] = (
                (1.0 - alpha) * base_va[:, cls_idx] +
                alpha * pred_va
            )

            modeled_counts[cls_idx] += 1

    score_base = macro_auc_skip_empty(y_true, oof_base_local)
    score_final = macro_auc_skip_empty(y_true, oof_final)

    return {
        "oof_base": oof_base_local,
        "oof_final": oof_final,
        "modeled_counts": modeled_counts,
        "score_base": score_base,
        "score_final": score_final,
    }


In [ ]:
# ProtoSSM v4 — Enhanced with Cross-Attention Layer

class SelectiveSSM(nn.Module):
    # Simplified Mamba-style selective state space model.
    # Input-dependent (selective) discretization of continuous-time SSM.
    # For T=12 bioacoustic windows, the sequential scan is efficient on CPU.

    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d = nn.Conv1d(
            d_model, d_model, d_conv,
            padding=d_conv - 1, groups=d_model
        )
        self.dt_proj = nn.Linear(d_model, d_model, bias=True)

        A = torch.arange(1, d_state + 1, dtype=torch.float32)
        A = A.unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B_size, T, D = x.shape
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)

        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)

        dt = F.softplus(self.dt_proj(x_conv))
        A = -torch.exp(self.A_log)
        B = self.B_proj(x_conv)
        C = self.C_proj(x_conv)

        h = torch.zeros(B_size, D, self.d_state, device=x.device)
        ys = []
        for t in range(T):
            dt_t = dt[:, t, :]
            dA = torch.exp(A[None, :, :] * dt_t[:, :, None])
            dB = dt_t[:, :, None] * B[:, t, None, :]
            h = h * dA + x[:, t, :, None] * dB
            y_t = (h * C[:, t, None, :]).sum(-1)
            ys.append(y_t)

        y = torch.stack(ys, dim=1)
        return y + x * self.D[None, None, :]


class TemporalCrossAttention(nn.Module):
    """Multi-head cross-attention between temporal windows.
    Captures non-local patterns (e.g., dawn chorus onset, counter-singing)
    that sequential SSM may miss."""
    
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x):
        # x: (B, T, D)
        residual = x
        x = self.norm(x)
        attn_out, _ = self.attn(x, x, x)
        x = residual + attn_out
        
        residual = x
        x = self.norm2(x)
        x = residual + self.ffn(x)
        return x


class ProtoSSMv2(nn.Module):
    # Prototypical State Space Model v4 with cross-attention and metadata awareness.
    #
    # V16 additions:
    # - Cross-attention layer after SSM for non-local temporal patterns
    # - All other v2 features preserved (metadata, prototypes, gated fusion)
    
    def __init__(self, d_input=1536, d_model=192, d_state=16,
                 n_ssm_layers=2, n_classes=234, n_windows=12,
                 dropout=0.2, n_sites=20, meta_dim=16,
                 use_cross_attn=True, cross_attn_heads=4):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.n_windows = n_windows

        # 1. Feature projection
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # 2. Learnable positional encoding
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)

        # 3. Metadata embeddings
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)

        # 4. Bidirectional SSM layers
        self.ssm_fwd = nn.ModuleList()
        self.ssm_bwd = nn.ModuleList()
        self.ssm_merge = nn.ModuleList()
        self.ssm_norm = nn.ModuleList()
        for _ in range(n_ssm_layers):
            self.ssm_fwd.append(SelectiveSSM(d_model, d_state))
            self.ssm_bwd.append(SelectiveSSM(d_model, d_state))
            self.ssm_merge.append(nn.Linear(2 * d_model, d_model))
            self.ssm_norm.append(nn.LayerNorm(d_model))
        self.ssm_drop = nn.Dropout(dropout)

        # 4b. NEW: Cross-attention after SSM
        self.use_cross_attn = use_cross_attn
        if use_cross_attn:
            self.cross_attn = TemporalCrossAttention(d_model, n_heads=cross_attn_heads, dropout=dropout)

        # 5. Learnable class prototypes
        self.prototypes = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp = nn.Parameter(torch.tensor(5.0))

        # 6. Per-class calibration bias
        self.class_bias = nn.Parameter(torch.zeros(n_classes))

        # 7. Per-class gated fusion with Perch logits
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))

        # 8. Taxonomic auxiliary head
        self.n_families = 0
        self.family_head = None

    def init_prototypes_from_data(self, embeddings, labels):
        with torch.no_grad():
            h = self.input_proj(embeddings)
            for c in range(self.n_classes):
                mask = labels[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def init_family_head(self, n_families, class_to_family):
        self.n_families = n_families
        self.family_head = nn.Linear(self.d_model, n_families)
        self.register_buffer('class_to_family', torch.tensor(class_to_family, dtype=torch.long))

    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape

        # Project embeddings
        h = self.input_proj(emb)
        h = h + self.pos_enc[:, :T, :]

        # Add metadata embeddings
        if site_ids is not None and hours is not None:
            s_emb = self.site_emb(site_ids)
            h_emb = self.hour_emb(hours)
            meta = self.meta_proj(torch.cat([s_emb, h_emb], dim=-1))
            h = h + meta[:, None, :]

        # Bidirectional SSM
        for fwd, bwd, merge, norm in zip(
            self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm
        ):
            residual = h
            h_f = fwd(h)
            h_b = bwd(h.flip(1)).flip(1)
            h = merge(torch.cat([h_f, h_b], dim=-1))
            h = self.ssm_drop(h)
            h = norm(h + residual)

        # NEW: Cross-attention for non-local temporal patterns
        if self.use_cross_attn:
            h = self.cross_attn(h)

        h_temporal = h

        # Prototypical cosine similarity + class bias
        h_norm = F.normalize(h, dim=-1)
        p_norm = F.normalize(self.prototypes, dim=-1)
        temp = F.softplus(self.proto_temp)
        sim = torch.matmul(h_norm, p_norm.T) * temp + self.class_bias[None, None, :]

        # Gated fusion with Perch logits
        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            species_logits = alpha * sim + (1 - alpha) * perch_logits
        else:
            species_logits = sim

        # Taxonomic auxiliary prediction
        family_logits = None
        if self.family_head is not None:
            h_pool = h.mean(dim=1)
            family_logits = self.family_head(h_pool)

        return species_logits, family_logits, h_temporal

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

ssm_cfg = CFG["proto_ssm"]
print("ProtoSSMv4 architecture defined (with cross-attention).")
test_model = ProtoSSMv2(
    d_model=ssm_cfg["d_model"], n_ssm_layers=2,
    n_sites=ssm_cfg["n_sites"], meta_dim=ssm_cfg["meta_dim"],
    use_cross_attn=ssm_cfg.get("use_cross_attn", True),
    cross_attn_heads=ssm_cfg.get("cross_attn_heads", 4),
)
print(f"Parameter count: {test_model.count_parameters():,}")
del test_model


In [ ]:
# ProtoSSM v4 Training Loop — with Mixup, Focal Loss, SWA

def build_taxonomy_groups(taxonomy_df, primary_labels):
    for col in ["family", "order", "class_name"]:
        if col in taxonomy_df.columns:
            group_map = taxonomy_df.set_index("primary_label")[col].to_dict()
            break
    else:
        group_map = {label: "Unknown" for label in primary_labels}

    groups = sorted(set(group_map.values()))
    grp_to_idx = {g: i for i, g in enumerate(groups)}
    class_to_group = []
    for label in primary_labels:
        grp = group_map.get(label, "Unknown")
        class_to_group.append(grp_to_idx.get(grp, 0))
    return len(groups), class_to_group, grp_to_idx


def build_site_mapping(meta_df):
    sites = meta_df["site"].unique().tolist()
    site_to_idx = {s: i + 1 for i, s in enumerate(sites)}
    n_sites = len(sites) + 1
    return site_to_idx, n_sites


def reshape_to_files(flat_array, meta_df, n_windows=N_WINDOWS):
    filenames = meta_df["filename"].to_numpy()
    unique_files = []
    seen = set()
    for f in filenames:
        if f not in seen:
            unique_files.append(f)
            seen.add(f)

    n_files = len(unique_files)
    assert len(flat_array) == n_files * n_windows, \
        f"Expected {n_files * n_windows} rows, got {len(flat_array)}"

    new_shape = (n_files, n_windows) + flat_array.shape[1:]
    return flat_array.reshape(new_shape), unique_files


def get_file_metadata(meta_df, file_list, site_to_idx, n_sites_max):
    file_to_row = {}
    filenames = meta_df["filename"].to_numpy()
    sites = meta_df["site"].to_numpy()
    hours = meta_df["hour_utc"].to_numpy()
    for i, f in enumerate(filenames):
        if f not in file_to_row:
            file_to_row[f] = i

    site_ids = np.zeros(len(file_list), dtype=np.int64)
    hour_ids = np.zeros(len(file_list), dtype=np.int64)
    for fi, fname in enumerate(file_list):
        row = file_to_row.get(fname)
        if row is not None:
            sid = site_to_idx.get(sites[row], 0)
            site_ids[fi] = min(sid, n_sites_max - 1)
            hour_ids[fi] = int(hours[row]) % 24
    return site_ids, hour_ids


def mixup_files(emb, logits, labels, site_ids, hours, families, alpha=0.3):
    """File-level mixup augmentation for ProtoSSM training.
    Mixes pairs of files with random lambda from Beta(alpha, alpha).
    Returns augmented versions of all inputs."""
    n = len(emb)
    if alpha <= 0 or n < 2:
        return emb, logits, labels, site_ids, hours, families
    
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1.0 - lam)  # Ensure lam >= 0.5 (dominant sample stays dominant)
    
    perm = np.random.permutation(n)
    
    emb_mix = lam * emb + (1 - lam) * emb[perm]
    logits_mix = lam * logits + (1 - lam) * logits[perm]
    labels_mix = lam * labels + (1 - lam) * labels[perm]
    
    # For discrete features (site, hour), keep the dominant sample's values
    families_mix = lam * families + (1 - lam) * families[perm] if families is not None else None
    
    return emb_mix, logits_mix, labels_mix, site_ids, hours, families_mix


def train_proto_ssm_single(model, emb_train, logits_train, labels_train,
                           site_ids_train=None, hours_train=None,
                           emb_val=None, logits_val=None, labels_val=None,
                           site_ids_val=None, hours_val=None,
                           file_families_train=None, file_families_val=None,
                           cfg=None, verbose=True):
    """Train a single ProtoSSM v4 model with mixup, focal loss, and SWA."""
    if cfg is None:
        cfg = CFG["proto_ssm_train"]

    label_smoothing = cfg.get("label_smoothing", 0.0)
    mixup_alpha = cfg.get("mixup_alpha", 0.0)
    focal_gamma = cfg.get("focal_gamma", 0.0)
    swa_start_frac = cfg.get("swa_start_frac", 1.0)  # 1.0 = disabled
    n_epochs = cfg["n_epochs"]
    swa_start_epoch = int(n_epochs * swa_start_frac)

    # Convert to tensors (base — unmixed)
    labels_np = labels_train.copy()
    
    # Apply label smoothing
    if label_smoothing > 0:
        labels_np = labels_np * (1.0 - label_smoothing) + label_smoothing / 2.0

    has_val = emb_val is not None
    if has_val:
        emb_v = to_device_tensor(emb_val, torch.float32)
        logits_v = to_device_tensor(logits_val, torch.float32)
        labels_v = to_device_tensor(labels_val, torch.float32)
        site_v = to_device_tensor(site_ids_val, torch.long) if site_ids_val is not None else None
        hour_v = to_device_tensor(hours_val, torch.long) if hours_val is not None else None

    fam_v = to_device_tensor(file_families_val, torch.float32) if (has_val and file_families_val is not None) else None

    # Class weights for imbalanced data
    labels_tr_t = to_device_tensor(labels_np, torch.float32)
    pos_counts = labels_tr_t.sum(dim=(0, 1))
    total = labels_tr_t.shape[0] * labels_tr_t.shape[1]
    pos_weight = ((total - pos_counts) / (pos_counts + 1)).clamp(max=cfg["pos_weight_cap"])

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg["lr"],
        epochs=n_epochs, steps_per_epoch=1,
        pct_start=0.1, anneal_strategy='cos'
    )

    best_val_loss = float('inf')
    best_state = None
    wait = 0
    history = {"train_loss": [], "val_loss": [], "val_auc": []}

    # SWA state accumulator
    swa_state = None
    swa_count = 0

    for epoch in range(n_epochs):
        # === Mixup augmentation (per-epoch re-sampling) ===
        if mixup_alpha > 0 and epoch > 5:  # Skip mixup for first 5 epochs (warmup)
            emb_mix, logits_mix, labels_mix, _, _, fam_mix = mixup_files(
                emb_train, logits_train, labels_np,
                site_ids_train, hours_train, file_families_train,
                alpha=mixup_alpha,
            )
        else:
            emb_mix, logits_mix, labels_mix = emb_train, logits_train, labels_np
            fam_mix = file_families_train

        emb_tr = to_device_tensor(emb_mix, torch.float32)
        logits_tr = to_device_tensor(logits_mix, torch.float32)
        labels_tr = to_device_tensor(labels_mix, torch.float32)
        site_tr = to_device_tensor(site_ids_train, torch.long) if site_ids_train is not None else None
        hour_tr = to_device_tensor(hours_train, torch.long) if hours_train is not None else None
        fam_tr = to_device_tensor(fam_mix, torch.float32) if fam_mix is not None else None

        # === Train ===
        model.train()
        with autocast_context():
            species_out, family_out, _ = model(emb_tr, logits_tr, site_ids=site_tr, hours=hour_tr)

        # Primary loss: focal BCE or weighted BCE
        if focal_gamma > 0:
            loss_main = focal_bce_with_logits(
                species_out, labels_tr,
                gamma=focal_gamma,
                pos_weight=pos_weight[None, None, :],
            )
        else:
            loss_main = F.binary_cross_entropy_with_logits(
                species_out, labels_tr,
                pos_weight=pos_weight[None, None, :]
            )

        # Knowledge distillation loss
        loss_distill = F.mse_loss(species_out, logits_tr)

        # Total loss
        loss = loss_main + cfg["distill_weight"] * loss_distill

        # Taxonomic auxiliary loss
        if family_out is not None and fam_tr is not None:
            loss_family = F.binary_cross_entropy_with_logits(family_out, fam_tr)
            loss = loss + 0.1 * loss_family

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        # === SWA accumulation ===
        if epoch >= swa_start_epoch:
            if swa_state is None:
                swa_state = {k: v.clone() for k, v in model.state_dict().items()}
                swa_count = 1
            else:
                for k in swa_state:
                    swa_state[k] += model.state_dict()[k]
                swa_count += 1

        # === Validate ===
        model.eval()
        with torch.no_grad():
            if has_val:
                val_out, val_fam, _ = model(emb_v, logits_v, site_ids=site_v, hours=hour_v)
                val_loss = F.binary_cross_entropy_with_logits(
                    val_out, labels_v,
                    pos_weight=pos_weight[None, None, :]
                )

                val_pred = tensor_to_numpy(val_out.reshape(-1, val_out.shape[-1]))
                val_true = tensor_to_numpy(labels_v.reshape(-1, labels_v.shape[-1]))
                try:
                    val_auc = macro_auc_skip_empty(val_true, val_pred)
                except Exception:
                    val_auc = 0.0
            else:
                val_loss = loss
                val_auc = 0.0

        history["train_loss"].append(loss.item())
        history["val_loss"].append(val_loss.item())
        history["val_auc"].append(val_auc)

        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if verbose and (epoch + 1) % 20 == 0:
            lr_now = optimizer.param_groups[0]['lr']
            swa_info = f" swa={swa_count}" if swa_count > 0 else ""
            print(f"  Epoch {epoch+1:3d}: train={loss.item():.4f} val={val_loss.item():.4f} "
                  f"auc={val_auc:.4f} lr={lr_now:.6f} wait={wait}{swa_info}")

        if wait >= cfg["patience"]:
            if verbose:
                print(f"  Early stopping at epoch {epoch+1} (best val_loss={best_val_loss:.4f})")
            break

    # Apply SWA if we accumulated enough checkpoints
    if swa_state is not None and swa_count >= 3:
        if verbose:
            print(f"  Applying SWA (averaged {swa_count} checkpoints)")
        avg_state = {k: v / swa_count for k, v in swa_state.items()}
        model.load_state_dict(avg_state)
    elif best_state is not None:
        model.load_state_dict(best_state)

    if verbose:
        print(f"  Training complete. Best val_loss={best_val_loss:.4f}")
        with torch.no_grad():
            alphas = torch.sigmoid(model.fusion_alpha).numpy()
            print(f"  Fusion alpha: mean={alphas.mean():.3f} min={alphas.min():.3f} max={alphas.max():.3f}")
            print(f"  Proto temperature: {F.softplus(model.proto_temp).item():.3f}")

    return model, history


def run_proto_ssm_oof(emb_files, logits_files, labels_files,
                      site_ids_all, hours_all,
                      file_families, file_groups,
                      n_families, class_to_family,
                      cfg=None, verbose=True):
    """Run GroupKFold OOF cross-validation for ProtoSSM v4."""
    if cfg is None:
        cfg = CFG["proto_ssm_train"]

    n_splits = cfg.get("oof_n_splits", 5)
    n_files = len(emb_files)
    ssm_cfg = CFG["proto_ssm"]

    oof_preds = np.zeros((n_files, N_WINDOWS, N_CLASSES), dtype=np.float32)
    fold_histories = []
    fold_alphas = []

    n_unique_groups = len(set(file_groups))
    if n_unique_groups < n_splits:
        print(f"  WARNING: Only {n_unique_groups} groups, reducing n_splits from {n_splits} to {n_unique_groups}")
        n_splits = n_unique_groups
    gkf = GroupKFold(n_splits=n_splits)
    dummy_y = np.zeros(n_files)

    for fold_i, (train_idx, val_idx) in enumerate(gkf.split(dummy_y, dummy_y, file_groups)):
        if verbose:
            print(f"\n--- Fold {fold_i+1}/{n_splits} (train={len(train_idx)}, val={len(val_idx)}) ---")

        fold_model = ProtoSSMv2(
            d_input=emb_files.shape[2],
            d_model=ssm_cfg["d_model"],
            d_state=ssm_cfg["d_state"],
            n_ssm_layers=ssm_cfg["n_ssm_layers"],
            n_classes=N_CLASSES,
            n_windows=N_WINDOWS,
            dropout=ssm_cfg["dropout"],
            n_sites=ssm_cfg["n_sites"],
            meta_dim=ssm_cfg["meta_dim"],
            use_cross_attn=ssm_cfg.get("use_cross_attn", True),
            cross_attn_heads=ssm_cfg.get("cross_attn_heads", 4),
        ).to(DEVICE)

        # Initialize prototypes
        emb_flat_fold = emb_files[train_idx].reshape(-1, emb_files.shape[2])
        labels_flat_fold = labels_files[train_idx].reshape(-1, N_CLASSES)
        fold_model.init_prototypes_from_data(
            to_device_tensor(emb_flat_fold, torch.float32),
            to_device_tensor(labels_flat_fold, torch.float32)
        )
        fold_model.init_family_head(n_families, class_to_family)

        # Train on fold
        fold_model, fold_hist = train_proto_ssm_single(
            fold_model,
            emb_files[train_idx], logits_files[train_idx], labels_files[train_idx].astype(np.float32),
            site_ids_train=site_ids_all[train_idx], hours_train=hours_all[train_idx],
            emb_val=emb_files[val_idx], logits_val=logits_files[val_idx],
            labels_val=labels_files[val_idx].astype(np.float32),
            site_ids_val=site_ids_all[val_idx], hours_val=hours_all[val_idx],
            file_families_train=file_families[train_idx],
            file_families_val=file_families[val_idx],
            cfg=cfg, verbose=verbose,
        )

        # OOF predictions with TTA
        fold_model.eval()
        tta_shifts = CFG.get("tta_shifts", [0])
        if len(tta_shifts) > 1:
            oof_preds[val_idx] = temporal_shift_tta(
                emb_files[val_idx], logits_files[val_idx], fold_model,
                site_ids_all[val_idx], hours_all[val_idx], shifts=tta_shifts
            )
        else:
            with torch.no_grad():
                val_emb = to_device_tensor(emb_files[val_idx], torch.float32)
                val_logits = to_device_tensor(logits_files[val_idx], torch.float32)
                val_sites = to_device_tensor(site_ids_all[val_idx], torch.long)
                val_hours = to_device_tensor(hours_all[val_idx], torch.long)
                with autocast_context():
                    val_out, _, _ = fold_model(val_emb, val_logits, site_ids=val_sites, hours=val_hours)
                oof_preds[val_idx] = tensor_to_numpy(val_out)

        fold_alphas.append(tensor_to_numpy(torch.sigmoid(fold_model.fusion_alpha)).copy())
        fold_histories.append(fold_hist)

    return oof_preds, fold_histories, fold_alphas


def optimize_ensemble_weight(oof_proto_flat, oof_mlp_flat, y_true_flat):
    """Grid search over blend weights to find optimal ProtoSSM ensemble weight."""
    weights = np.arange(0.0, 1.05, 0.05)
    results = []

    for w in weights:
        blended = w * oof_proto_flat + (1.0 - w) * oof_mlp_flat
        try:
            auc = macro_auc_skip_empty(y_true_flat, blended)
        except Exception:
            auc = 0.0
        results.append((w, auc))

    best_w, best_auc = max(results, key=lambda x: x[1])
    return best_w, best_auc, results


print("ProtoSSM v4 training functions defined (with mixup, focal loss, SWA, TTA).")


## Load V18 Artifacts
Load pretrained downstream components so submit mode does only hidden-test inference.


In [ ]:
# Artifact load cell
manifest = json.loads((ARTIFACTS_DIR / "artifacts_manifest.json").read_text())
with open(ARTIFACTS_DIR / manifest["artifact_files"]["prior_tables"], "rb") as f:
    final_prior_tables = pickle.load(f)
with open(ARTIFACTS_DIR / manifest["artifact_files"]["sklearn"], "rb") as f:
    sk_artifacts = pickle.load(f)

emb_scaler = sk_artifacts["emb_scaler"]
emb_pca = sk_artifacts["emb_pca"]
probe_models = sk_artifacts["probe_models"]
PER_CLASS_THRESHOLDS = np.load(ARTIFACTS_DIR / manifest["artifact_files"]["thresholds"]).astype(np.float32)

CALIBRATORS = []
CALIBRATION_MANIFEST = manifest.get("calibration", {})
calibrator_name = manifest.get("artifact_files", {}).get("calibrators")
if calibrator_name:
    with open(ARTIFACTS_DIR / calibrator_name, "rb") as f:
        CALIBRATORS = pickle.load(f)
calibration_manifest_name = manifest.get("artifact_files", {}).get("calibration_manifest")
if calibration_manifest_name and (ARTIFACTS_DIR / calibration_manifest_name).exists():
    CALIBRATION_MANIFEST = json.loads((ARTIFACTS_DIR / calibration_manifest_name).read_text())

BEST_PROBE = manifest["best_probe"]
ENSEMBLE_WEIGHT_PROTO = float(manifest["ensemble_weight_proto"])
CORRECTION_WEIGHT = float(manifest.get("correction_weight", 0.0))
site_to_idx = {str(k): int(v) for k, v in manifest["site_to_idx"].items()}
class_to_family = [int(x) for x in manifest["class_to_family"]]
fam_to_idx = {str(k): int(v) for k, v in manifest["fam_to_idx"].items()}
n_families = int(manifest["n_families"])
n_sites_cfg = int(manifest["n_sites_cfg"])

CFG["proxy_reduce"] = manifest["proxy_reduce"]
CFG["temperature"] = manifest["temperature"]
CFG["file_level_top_k"] = int(manifest.get("file_level_top_k", 0))
CFG["tta_shifts"] = [int(x) for x in manifest.get("tta_shifts", [0])]
CFG["rank_aware_scale"] = bool(manifest.get("rank_aware_scale", False))
CFG["rank_aware_power"] = float(manifest.get("rank_aware_power", 0.5))
CFG["delta_shift_alpha"] = float(manifest.get("delta_shift_alpha", 0.0))
CFG["proto_ssm"] = manifest.get("proto_ssm", CFG["proto_ssm"])
CFG["residual_ssm"] = manifest.get("residual_ssm", CFG["residual_ssm"])

proto_ckpt = torch.load(ARTIFACTS_DIR / manifest["artifact_files"]["proto"], map_location=DEVICE)
proto_cfg = proto_ckpt["proto_ssm"]
CFG["proto_ssm"] = proto_cfg
model = ProtoSSMv2(
    d_input=int(emb_pca.mean_.shape[0]) if hasattr(emb_pca, 'mean_') else 1536,
    d_model=proto_cfg["d_model"],
    d_state=proto_cfg["d_state"],
    n_ssm_layers=proto_cfg["n_ssm_layers"],
    n_classes=int(manifest["n_classes"]),
    n_windows=int(manifest["n_windows"]),
    dropout=proto_cfg["dropout"],
    n_sites=proto_cfg["n_sites"],
    meta_dim=proto_cfg["meta_dim"],
    use_cross_attn=proto_cfg.get("use_cross_attn", True),
    cross_attn_heads=proto_cfg.get("cross_attn_heads", 4),
).to(DEVICE)
model.init_family_head(n_families, class_to_family)
model.load_state_dict(proto_ckpt["state_dict"], strict=True)
model.eval()

class ResidualSSM(nn.Module):
    def __init__(self, d_input=1536, d_scores=234, d_model=64, d_state=8,
                 n_classes=234, n_windows=12, dropout=0.1, n_sites=20, meta_dim=8):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.input_proj = nn.Sequential(
            nn.Linear(d_input + d_scores, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.ssm_fwd = SelectiveSSM(d_model, d_state)
        self.ssm_bwd = SelectiveSSM(d_model, d_state)
        self.ssm_merge = nn.Linear(2 * d_model, d_model)
        self.ssm_norm = nn.LayerNorm(d_model)
        self.ssm_drop = nn.Dropout(dropout)
        self.output_head = nn.Linear(d_model, n_classes)
        nn.init.zeros_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)

    def forward(self, emb, first_pass_scores, site_ids=None, hours=None):
        B, T, _ = emb.shape
        x = torch.cat([emb, first_pass_scores], dim=-1)
        h = self.input_proj(x)
        if site_ids is not None and hours is not None:
            site_e = self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1))
            hour_e = self.hour_emb(hours.clamp(0, 23))
            meta = self.meta_proj(torch.cat([site_e, hour_e], dim=-1))
            h = h + meta.unsqueeze(1)
        h = h + self.pos_enc[:, :T, :]
        residual = h
        h_f = self.ssm_fwd(h)
        h_b = self.ssm_bwd(h.flip(1)).flip(1)
        h = self.ssm_merge(torch.cat([h_f, h_b], dim=-1))
        h = self.ssm_drop(h)
        h = self.ssm_norm(h + residual)
        return self.output_head(h)

res_model = None
if manifest.get("has_residual", False) and manifest["artifact_files"].get("residual"):
    res_ckpt = torch.load(ARTIFACTS_DIR / manifest["artifact_files"]["residual"], map_location=DEVICE)
    res_cfg = res_ckpt["residual_ssm"]
    CFG["residual_ssm"] = res_cfg
    res_model = ResidualSSM(
        d_input=int(emb_pca.mean_.shape[0]) if hasattr(emb_pca, 'mean_') else 1536,
        d_scores=int(manifest["n_classes"]),
        d_model=res_cfg["d_model"],
        d_state=res_cfg["d_state"],
        n_classes=int(manifest["n_classes"]),
        n_windows=int(manifest["n_windows"]),
        dropout=res_cfg["dropout"],
        n_sites=proto_cfg["n_sites"],
        meta_dim=8,
    ).to(DEVICE)
    res_model.load_state_dict(res_ckpt["state_dict"], strict=True)
    res_model.eval()

artifact_primary_labels = list(manifest["primary_labels"])
if artifact_primary_labels != list(PRIMARY_LABELS):
    raise ValueError("Artifact primary-label order does not match notebook taxonomy order")
N_CLASSES = int(manifest["n_classes"])
N_WINDOWS = int(manifest["n_windows"])
if N_CLASSES != len(PRIMARY_LABELS):
    raise ValueError(f"Artifact class count mismatch: {N_CLASSES} vs {len(PRIMARY_LABELS)}")
print("Loaded artifactized V18 calibration-refresh stack.")
print("  Proto params:", manifest["proto_parameters"])
print("  Residual enabled:", manifest.get("has_residual", False))
print("  Probe models:", len(probe_models))
print("  Calibrated classes:", int(sum(cal is not None for cal in CALIBRATORS)) if CALIBRATORS else 0)


In [ ]:
# Cell 16 — Infer Perch on hidden test with embeddings
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))

if len(test_paths) == 0:
    print(f"Hidden test not mounted. Dry-run on first {CFG['dryrun_n_files']} train soundscapes.")
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:CFG["dryrun_n_files"]]
else:
    print(f"Hidden test files: {len(test_paths)}")

# [MODIFIED - Opsi 3] Gunakan proxy_reduce terbaik dari grid search (bukan hardcode "max")
meta_test, scores_test_raw, emb_test = infer_perch_with_embeddings(
    test_paths,
    batch_files=CFG["batch_files"],
    verbose=CFG["verbose"],
    proxy_reduce=CFG["proxy_reduce"],  # hasil grid search, default "max"
)
print(f"proxy_reduce used for test inference: {CFG['proxy_reduce']!r}")

print("meta_test:", meta_test.shape)
print("scores_test_raw:", scores_test_raw.shape)
print("emb_test:", emb_test.shape)

record_stage_event("perch_hidden_test_inference_complete", n_files=int(len(test_paths)), n_rows=int(len(meta_test)))


In [ ]:
# Score Fusion: ProtoSSM v4 + MLP Probes + Priors + TTA (OOF-optimized weight)

# --- Step 1: ProtoSSM v4 inference on test with TTA ---
emb_test_files, test_file_list = reshape_to_files(emb_test, meta_test)
logits_test_files, _ = reshape_to_files(scores_test_raw, meta_test)

# Build test metadata
test_site_ids, test_hours = get_file_metadata(meta_test, test_file_list, site_to_idx, CFG["proto_ssm"]["n_sites"])

emb_test_tensor = to_device_tensor(emb_test_files, torch.float32)
logits_test_tensor = to_device_tensor(logits_test_files, torch.float32)
test_site_tensor = to_device_tensor(test_site_ids, torch.long)
test_hour_tensor = to_device_tensor(test_hours, torch.long)

# V16: TTA — average predictions from shifted temporal sequences
model.eval()
tta_shifts = CFG.get("tta_shifts", [0])
if len(tta_shifts) > 1:
    print(f"Running TTA with shifts: {tta_shifts}")
    proto_scores = temporal_shift_tta(
        emb_test_files, logits_test_files, model,
        test_site_ids, test_hours, shifts=tta_shifts
    )
else:
    with torch.no_grad():
        with autocast_context():
            proto_out, _, h_test = model(emb_test_tensor, logits_test_tensor,
                                          site_ids=test_site_tensor, hours=test_hour_tensor)
        proto_scores = tensor_to_numpy(proto_out)

# Flatten back to (n_rows, n_classes)
proto_scores_flat = proto_scores.reshape(-1, N_CLASSES).astype(np.float32)

print(f"ProtoSSM v4 test scores: {proto_scores_flat.shape}")
print(f"Score range: {proto_scores_flat.min():.3f} to {proto_scores_flat.max():.3f}")

# --- Step 2: Prior-fused base scores ---
test_base_scores, test_prior_scores = fuse_scores_with_tables(
    scores_test_raw,
    sites=meta_test["site"].to_numpy(),
    hours=meta_test["hour_utc"].to_numpy(),
    tables=final_prior_tables,
)

# --- Step 3: MLP probe scores ---
emb_test_scaled = emb_scaler.transform(emb_test)
Z_TEST = emb_pca.transform(emb_test_scaled).astype(np.float32)

mlp_scores = test_base_scores.copy()

for cls_idx, clf in probe_models.items():
    X_cls_test = build_class_features(
        Z_TEST,
        raw_col=scores_test_raw[:, cls_idx],
        prior_col=test_prior_scores[:, cls_idx],
        base_col=test_base_scores[:, cls_idx],
    )

    if hasattr(clf, "predict_proba"):
        prob = clf.predict_proba(X_cls_test)[:, 1].astype(np.float32)
        pred = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
    else:
        pred = clf.decision_function(X_cls_test).astype(np.float32)

    alpha = float(CFG["frozen_best_probe"]["alpha"])
    mlp_scores[:, cls_idx] = (1.0 - alpha) * test_base_scores[:, cls_idx] + alpha * pred

# --- Step 4: Ensemble fusion with OOF-optimized weight ---
print(f"\nUsing OOF-optimized ensemble weight: {ENSEMBLE_WEIGHT_PROTO:.2f}")

final_test_scores = (
    ENSEMBLE_WEIGHT_PROTO * proto_scores_flat +
    (1.0 - ENSEMBLE_WEIGHT_PROTO) * mlp_scores
).astype(np.float32)

# --- Step 5: Residual SSM correction (second pass) ---
if res_model is not None and CORRECTION_WEIGHT > 0:
    first_pass_test_files, _ = reshape_to_files(final_test_scores, meta_test)
    first_pass_test_t = to_device_tensor(first_pass_test_files, torch.float32)

    res_model.eval()
    with torch.no_grad():
        with autocast_context():
            test_correction = res_model(
                emb_test_tensor, first_pass_test_t,
                site_ids=test_site_tensor, hours=test_hour_tensor
            )
        test_correction = tensor_to_numpy(test_correction)

    test_correction_flat = test_correction.reshape(-1, N_CLASSES).astype(np.float32)

    print(f"\nResidual correction: mean_abs={np.abs(test_correction_flat).mean():.4f}, "
          f"max={np.abs(test_correction_flat).max():.4f}")

    final_test_scores = final_test_scores + CORRECTION_WEIGHT * test_correction_flat
    print(f"Final scores (after residual): range [{final_test_scores.min():.3f}, {final_test_scores.max():.3f}]")
else:
    print("\nResidual correction: SKIPPED")

print(f"Final scores: {final_test_scores.shape}")

# --- Logging ---
test_logs = {}
window_scores = proto_scores.reshape(-1, N_WINDOWS, N_CLASSES).mean(axis=(0, 2))
test_logs["window_position_scores"] = window_scores.tolist()
print(f"\nWindow position mean scores: {[f'{s:.3f}' for s in window_scores]}")

if hasattr(model, 'class_to_family'):
    taxon_scores = defaultdict(list)
    idx_to_fam = {v: k for k, v in fam_to_idx.items()}
    for ci in range(N_CLASSES):
        fam_idx = class_to_family[ci]
        fam_name = idx_to_fam.get(fam_idx, f"group_{fam_idx}")
        taxon_scores[fam_name].append(float(proto_scores_flat[:, ci].mean()))

    test_logs["taxon_mean_scores"] = {k: float(np.mean(v)) for k, v in taxon_scores.items()}
    for k, v in sorted(taxon_scores.items(), key=lambda x: -np.mean(x[1]))[:5]:
        print(f"  {k}: mean_score={np.mean(v):.4f} (n_classes={len(v)})")

with torch.no_grad():
    p_norm = F.normalize(model.prototypes, dim=-1)
    cos_sim = torch.matmul(p_norm, p_norm.T)
    cos_sim.fill_diagonal_(0)
    top_sims = tensor_to_numpy(cos_sim.max(dim=1)[0])
    test_logs["prototype_max_similarity"] = {
        "mean": float(top_sims.mean()),
        "max": float(top_sims.max()),
        "min": float(top_sims.min()),
    }
    print(f"\nPrototype nearest-neighbor similarity: mean={top_sims.mean():.3f}, max={top_sims.max():.3f}")

LOGS["test_inference"] = test_logs
record_stage_event("stack_replay_complete", n_rows=int(final_test_scores.shape[0]), n_classes=int(final_test_scores.shape[1]))


In [ ]:
# Cell 18 — V18: Full post-processing pipeline

# Artifactized per-class thresholds
if MODE == "train" and oof_proto_flat is not None:
    print("Optimizing per-class thresholds from OOF...")
    best_thresholds, best_scores = optimize_per_class_thresholds(
        oof_proto_flat, Y_FULL, n_windows=N_WINDOWS, thresholds=CFG["threshold_grid"]
    )
    PER_CLASS_THRESHOLDS = best_thresholds.astype(np.float32)
    print(f"  Mean threshold: {best_thresholds.mean():.3f}")
    print(f"  Threshold range: [{best_thresholds.min():.2f}, {best_thresholds.max():.2f}]")
    print(f"  Mean F1 (proxy): {best_scores.mean():.3f}")
else:
    print(f"Using artifact per-class thresholds: mean={PER_CLASS_THRESHOLDS.mean():.3f}, range=[{PER_CLASS_THRESHOLDS.min():.2f}, {PER_CLASS_THRESHOLDS.max():.2f}]")


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

current_row_ids = meta_test["row_id"].values

# --- Step 1: Per-taxon temperature scaling ---
temp_cfg = CFG["temperature"]
T_AVES = temp_cfg["aves"]
T_TEXTURE = temp_cfg["texture"]

class_temperatures = np.ones(N_CLASSES, dtype=np.float32) * T_AVES
for ci, label in enumerate(PRIMARY_LABELS):
    cn = CLASS_NAME_MAP.get(label, "Aves")
    if cn in TEXTURE_TAXA:
        class_temperatures[ci] = T_TEXTURE

print(f"\nPer-taxon temperature: Aves={T_AVES}, Texture={T_TEXTURE}")

scaled_scores = final_test_scores / class_temperatures[None, :]
probs = sigmoid(scaled_scores)
record_stage_event("temperature_scaling_complete")
if TIMER_CFG["save_early_snapshots"]:
    submission = save_submission_snapshot(current_row_ids, probs, "after_temperature", make_primary=True)

# --- Step 1b: Isotonic calibration (artifactized) ---
def apply_file_level_calibrators(scores, calibrators, n_windows=12):
    if not calibrators:
        return scores, {"used_classes": 0, "scale_mean": 1.0, "scale_std": 0.0}
    scores_files = scores.reshape(-1, n_windows, scores.shape[1]).copy()
    file_max = scores_files.max(axis=1)
    calibrated_max = file_max.copy()
    used = 0
    for ci, cal in enumerate(calibrators):
        if cal is None:
            continue
        calibrated_max[:, ci] = np.clip(cal.transform(file_max[:, ci]), 0.0, 1.0)
        used += 1
    scale = calibrated_max / np.clip(file_max, 1e-6, None)
    scale = np.clip(scale, 0.25, 4.0)
    scores_files *= scale[:, None, :]
    stats = {
        "used_classes": int(used),
        "scale_mean": float(scale.mean()),
        "scale_std": float(scale.std()),
    }
    return np.clip(scores_files.reshape(scores.shape), 0.0, 1.0), stats

if CALIBRATORS:
    req = TIMER_CFG["stage_requirements"].get("isotonic_calibration", 0.0)
    if timer_allows("isotonic_calibration", req):
        probs, cal_stats = apply_file_level_calibrators(probs, CALIBRATORS, n_windows=N_WINDOWS)
        print(f"Applied isotonic calibration to {cal_stats['used_classes']} classes; scale mean={cal_stats['scale_mean']:.3f}, std={cal_stats['scale_std']:.3f}")
        LOGS["calibration_stats"] = cal_stats
        record_stage_event("isotonic_calibration_complete", used_classes=int(cal_stats["used_classes"]))
        if TIMER_CFG["save_early_snapshots"]:
            submission = save_submission_snapshot(current_row_ids, probs, "after_calibration", make_primary=True)
    else:
        print("Skipping isotonic calibration due to timer budget")
        LOGS["calibration_stats"] = {"skipped_due_timer": True, "used_classes": 0}
else:
    print("No calibration artifacts found; using identity calibration")

# --- Step 2: File-level confidence scaling ---
top_k = CFG.get("file_level_top_k", 0)
if top_k > 0:
    req = TIMER_CFG["stage_requirements"].get("file_level_scaling", 0.0)
    if timer_allows("file_level_scaling", req):
        print(f"Applying file-level confidence scaling (top_k={top_k})")
        probs = file_level_confidence_scale(probs, n_windows=N_WINDOWS, top_k=top_k)
        probs = np.clip(probs, 0.0, 1.0)
        record_stage_event("file_level_scaling_complete", top_k=int(top_k))
        if TIMER_CFG["save_early_snapshots"]:
            submission = save_submission_snapshot(current_row_ids, probs, "after_file_level_scaling", make_primary=True)
    else:
        print("Skipping file-level confidence scaling due to timer budget")

# --- Step 3: V17 Rank-aware post-processing ---
if CFG.get("rank_aware_scale", False):
    req = TIMER_CFG["stage_requirements"].get("rank_aware_scaling", 0.0)
    if timer_allows("rank_aware_scaling", req):
        power = CFG.get("rank_aware_power", 0.5)
        print(f"Applying rank-aware scaling (power={power})")
        probs = rank_aware_scaling(probs, n_windows=N_WINDOWS, power=power)
        probs = np.clip(probs, 0.0, 1.0)
        record_stage_event("rank_aware_scaling_complete", power=float(power))
        if TIMER_CFG["save_early_snapshots"]:
            submission = save_submission_snapshot(current_row_ids, probs, "after_rank_scaling", make_primary=True)
    else:
        print("Skipping rank-aware scaling due to timer budget")

# --- Step 4: V18 Adaptive delta shift smoothing ---
def adaptive_delta_smooth(probs, n_windows, base_alpha=0.20):
    n_files = probs.shape[0] // n_windows
    result = probs.copy()
    view = result.reshape(n_files, n_windows, -1)
    p_view = probs.reshape(n_files, n_windows, -1)
    for i in range(1, n_windows - 1):
        conf = p_view[:, i, :].max(axis=-1, keepdims=True)
        a = base_alpha * (1.0 - conf)
        neighbor_avg = (p_view[:, i - 1, :] + p_view[:, i + 1, :]) / 2.0
        view[:, i, :] = (1.0 - a) * p_view[:, i, :] + a * neighbor_avg
    return result.reshape(probs.shape)

alpha = CFG.get("delta_shift_alpha", 0.0)
if alpha > 0:
    req = TIMER_CFG["stage_requirements"].get("adaptive_delta_smooth", 0.0)
    if timer_allows("adaptive_delta_smooth", req):
        print(f"Applying adaptive delta shift smoothing (alpha={alpha})")
        probs = adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=alpha)
        probs = np.clip(probs, 0.0, 1.0)
        record_stage_event("adaptive_delta_smooth_complete", alpha=float(alpha))
        if TIMER_CFG["save_early_snapshots"]:
            submission = save_submission_snapshot(current_row_ids, probs, "after_delta_smooth", make_primary=True)
    else:
        print("Skipping adaptive delta shift smoothing due to timer budget")

# --- Step 5: V18 Per-class threshold sharpening ---
req = TIMER_CFG["stage_requirements"].get("threshold_sharpening", 0.0)
if timer_allows("threshold_sharpening", req):
    print(f"Applying per-class threshold sharpening...")
    probs = apply_per_class_thresholds(probs, PER_CLASS_THRESHOLDS, n_windows=N_WINDOWS)
    record_stage_event("threshold_sharpening_complete")
    if TIMER_CFG["save_early_snapshots"]:
        submission = save_submission_snapshot(current_row_ids, probs, "after_thresholds", make_primary=True)
else:
    print("Skipping per-class threshold sharpening due to timer budget")

# --- Build final submission ---
submission = build_submission_df(current_row_ids, probs)
expected_rows = len(test_paths) * N_WINDOWS
assert len(submission) == expected_rows, f"Expected {expected_rows}, got {len(submission)}"
assert submission.columns.tolist() == ["row_id"] + PRIMARY_LABELS
assert not submission.isna().any().any()

submission.to_csv("submission.csv", index=False)
record_stage_event("final_submission_saved", rows=int(len(submission)))

print("\nSaved submission.csv")
print("Submission shape:", submission.shape)
print(f"Final score range: {probs.min():.6f} to {probs.max():.6f}")
print(f"Final mean: {probs.mean():.4f}")
print(submission.iloc[:3, :8])


In [ ]:
# Cell 19 — Final Diagnostics and Logging

# Save comprehensive logs
wall_time = time.time() - _WALL_START
LOGS["wall_time_seconds"] = wall_time
LOGS["temperature"] = CFG["temperature"]
LOGS["ensemble_weight_proto"] = ENSEMBLE_WEIGHT_PROTO
LOGS["n_classes"] = N_CLASSES
LOGS["n_windows"] = N_WINDOWS
LOGS["cfg_proto_ssm"] = CFG["proto_ssm"]
LOGS["cfg_proto_ssm_train"] = {k: v for k, v in CFG["proto_ssm_train"].items() if not isinstance(v, (np.ndarray,))}
LOGS["v18_improvements"] = [
    "d_model_320", "n_ssm_layers_4", "n_prototypes_2", "meta_dim_24",
    "cross_attention_heads_8", "per_taxon_temperature", "isotonic_refresh_calibration",
    "file_level_scaling", "tta_disabled", "rank_aware_scaling_v18",
    "adaptive_delta_smooth", "updated_probe_defaults", "updated_fusion_lambdas",
    "per_class_thresholds"
]
LOGS["per_class_thresholds"] = PER_CLASS_THRESHOLDS.tolist()
LOGS["calibration_manifest"] = CALIBRATION_MANIFEST
LOGS["timer_config"] = TIMER_CFG

try:
    with open("/kaggle/working/v18_timed_refresh_submit_logs.json", "w") as f:
        json.dump(LOGS, f, indent=2, default=str)
    print("Saved /kaggle/working/v18_timed_refresh_submit_logs.json")
except Exception as e:
    print(f"Warning: could not save logs: {e}")

if MODE == "train":
    print("=== ProtoSSM v5 Training Summary (V18 branch) ===")
    print(f"Parameters: {model.count_parameters():,}")
    print(f"d_model: {CFG['proto_ssm']['d_model']}, n_ssm_layers: {CFG['proto_ssm']['n_ssm_layers']}")
    print(f"Wall time: {wall_time:.1f}s")
    print(f"OOF CV time: {LOGS.get('oof_time', 0):.1f}s")
    print(f"Final model training time: {LOGS.get('train_time_final', 0):.1f}s")
    print(f"Final train loss: {train_history['train_loss'][-1]:.4f}")
    print(f"Best val loss: {min(train_history['val_loss']):.4f}")
    print(f"Best val AUC: {max(train_history['val_auc']):.4f}")

    print(f"\n=== OOF Results ===")
    print(f"ProtoSSM OOF AUC: {LOGS.get('oof_auc_proto', 0):.4f}")
    print(f"MLP-only OOF AUC: {LOGS.get('mlp_only_auc', 0):.4f}")
    print(f"Ensemble OOF AUC: {LOGS.get('ensemble_auc', 0):.4f}")
    print(f"Optimized ProtoSSM weight: {ENSEMBLE_WEIGHT_PROTO:.2f}")

    with torch.no_grad():
        alphas = tensor_to_numpy(torch.sigmoid(model.fusion_alpha))
        high_proto = (alphas > 0.5).sum()
        high_perch = (alphas <= 0.5).sum()
        print(f"\nFusion alpha distribution (final model):")
        print(f"  ProtoSSM-dominant (alpha>0.5): {high_proto} classes")
        print(f"  Perch-dominant (alpha<=0.5): {high_perch} classes")

    print(f"\nPer-class calibration bias stats:")
    with torch.no_grad():
        cb = tensor_to_numpy(model.class_bias)
        print(f"  mean={cb.mean():.4f} std={cb.std():.4f} min={cb.min():.4f} max={cb.max():.4f}")

    print(f"\nMLP probes: {len(probe_models)} classes")

    if "per_class_auc_proto" in LOGS and LOGS["per_class_auc_proto"]:
        sorted_aucs = sorted(LOGS["per_class_auc_proto"].items(), key=lambda x: x[1], reverse=True)
        print(f"\nTop 10 classes by ProtoSSM OOF AUC:")
        for label, auc in sorted_aucs[:10]:
            print(f"  {label}: {auc:.4f}")
        print(f"\nBottom 10 classes by ProtoSSM OOF AUC:")
        for label, auc in sorted_aucs[-10:]:
            print(f"  {label}: {auc:.4f}")

    print("\nSubmission probability stats:")
    print(submission.iloc[:, 1:].stack().describe())
else:
    print("Timed calibration-refresh artifactized submit mode completed.")
    print(f"ProtoSSM v5 parameters: {model.count_parameters():,}")
    print(f"Ensemble weight: {ENSEMBLE_WEIGHT_PROTO:.2f}")
    print(f"Wall time: {wall_time:.1f}s")
    print(f"V18 improvements: {LOGS['v18_improvements']}")
